# Contextual Extension: LinUCB vs Thompson Sampling (E8b)

Онлайн-сравнение на контекстной синтетике и кривые cumulative regret.

In [ ]:
import numpy as np

from src.bandits.linucb import LinUCBPolicy
from src.bandits.thompson_sampling import ThompsonSamplingPolicy

linucb = LinUCBPolicy(n_arms=3, context_dim=2, alpha=1.0, seed=42)
ts = ThompsonSamplingPolicy(n_arms=3, seed=42)

contexts = [np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([1.0, 1.0])]
for context in contexts * 20:
    lin_arm = linucb.select_arm(context)
    ts_arm = ts.select_arm()
    # Игрушечная reward-модель: arm 0 лучше при context[0], arm 1 лучше при context[1].
    reward_lin = float((lin_arm == 0 and context[0] > context[1]) or (lin_arm == 1 and context[1] > context[0]))
    reward_ts = float(ts_arm == 0)
    linucb.update(lin_arm, reward_lin, context)
    ts.update(ts_arm, reward_ts)

linucb.snapshot(), ts.snapshot()

In [ ]:
# --- E8b regret curves (load full run or re-run) ---
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.evaluation.plotting import plot_metric_by_policy
from src.experiments.configs import ExperimentConfig
from src.experiments.runner import compare_policies

results_path = Path("../outputs/extended_full/e8b_contextual_synthetic/results.csv")
if not results_path.exists():
    results_path = Path("outputs/extended_full/e8b_contextual_synthetic/results.csv")

if results_path.exists():
    e8b_results = pd.read_csv(results_path)
else:
    from src.bandits.fixed_ab import FixedABPolicy
    from src.bandits.linucb import LinUCBPolicy
    from src.bandits.thompson_sampling import ThompsonSamplingPolicy
    from src.environments.contextual_bernoulli import ContextualBernoulliBanditEnv

    e8b_configs = []
    for seed in range(10):
        for policy_name, factory in {
            "fixed_ab": lambda n_arms, s: FixedABPolicy(n_arms=n_arms, seed=s),
            "thompson_sampling": lambda n_arms, s: ThompsonSamplingPolicy(n_arms=n_arms, seed=s),
            "linucb": lambda n_arms, s: LinUCBPolicy(n_arms=n_arms, context_dim=4, alpha=1.0, seed=s),
        }.items():
            e8b_configs.append(
                ExperimentConfig(
                    run_id=f"{policy_name}_seed{seed}",
                    seed=seed,
                    horizon=10_000,
                    policy_factory=factory,
                    environment_factory=lambda s: ContextualBernoulliBanditEnv(
                        n_arms=5, context_dim=4, horizon=10_000, seed=s
                    ),
                )
            )
    e8b_results = compare_policies(e8b_configs)

plot_metric_by_policy(e8b_results, "cumulative_regret")
plt.suptitle("E8b: mean cumulative regret (contextual synthetic)")
plt.show()

In [ ]:
from IPython.display import Image, display
from pathlib import Path

path = Path("../outputs/figures/fig_3_1_e8b_regret_curves.png")
if path.exists():
    display(Image(filename=str(path), width=700))